# 01 — Raw Data Diagnosis & Cleaning Pipeline

## Overview
**Đồ án môn học DS108: Tiền xử lý và Xây dựng Bộ dữ liệu.**

Tài liệu này được thiết kế theo cấu trúc hai phần để thể hiện tư duy phương pháp luận rõ ràng:
1. **Phần 1: Khai phá Dữ liệu Thô (Diagnostic EDA)** — Khảo sát tập dữ liệu thô để phát hiện, đo lường và chứng minh các lỗi hệ thống, nhược điểm chí mạng (như lỗi đơn vị giá, khuyết thiếu hệ thống playtime, bias ratings do mẫu nhỏ).
2. **Phần 2: Quy trình Tiền xử lý & Làm sạch (Curation Pipeline)** — Thực thi thiết kế đường ống xử lý tự động để giải quyết các vấn đề đã phát hiện ở Phần 1, tạo ra tập dữ liệu thành phẩm 42 cột đạt chuẩn.

In [1]:
import pandas as pd
import numpy as np
import ast
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
RAW_PATH = os.path.join(BASE_DIR, 'data', 'raw', 'steam_games_raw.csv')
PROCESSED_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'steam_games_with_genres.csv')

print(f'Loading raw data from: {RAW_PATH}')
df_raw = pd.read_csv(RAW_PATH)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

Loading raw data from: D:\DS108_final\data\raw\steam_games_raw.csv
Raw shape: (10284, 25)


,appid,name,developer_x,publisher_x,score_rank,positive,negative,userscore,owners,average_forever,...,discount,ccu,store_name,release_date,genres,developer_y,publisher_y,store_price,is_free,year
0,1623730,Palworld,Pocketpair,Pocketpair,NaN,358266,22443,0,"50,000,000 .. 100,000,000",0,...,0,18028,Palworld,2024-01-18,"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",Pocketpair,Pocketpair,38500000.0,False,2024.0
1,1938090,Call of Duty: Modern Warfare II,"Treyarch, Raven Software, Beenox, High Moon St...",Activision,NaN,419594,294520,0,"50,000,000 .. 100,000,000",0,...,45,67419,Call of Duty®,2022-10-27,['Action'],Treyarch,Activision,NaN,False,2022.0
2,2358720,Black Myth: Wukong,Game Science,Game Science,NaN,1111720,38378,0,"50,000,000 .. 100,000,000",0,...,0,15004,Black Myth: Wukong,2024-08-19,"['Action', 'Adventure', 'RPG']",Game Science,Game Science,5999.0,False,2024.0


## Phần 1: Khai phá Dữ liệu Thô để Phát hiện Vấn đề (Diagnostic EDA)

Chúng ta tiến hành phân tích tập dữ liệu thô để phát hiện và làm rõ 5 vấn đề cốt lõi của dữ liệu:

In [2]:
print('=== 1. PRICE UNIT BUG (CENTS vs USD) ===')
# Check the raw price and initialprice summary stats
print(df_raw[['price', 'initialprice']].describe())
print(f"Sample raw initialprice values: {df_raw['initialprice'].head(5).tolist()}")
print('Nhận xét: Giá trị max của initialprice lên đến 19,999.00 USD — lạm phát 100 lần do Store API lưu dạng cents (xu) thay vì USD.')

print('\n=== 2. PLAYTIME SYSTEMATIC MISSINGNESS (MNAR) ===')
# Extract year from release_date dynamically to group
df_raw['year_temp'] = pd.to_datetime(df_raw['release_date'], errors='coerce').dt.year
playtime_by_year = df_raw.groupby('year_temp')[['average_forever', 'median_forever']].mean()
print(playtime_by_year)
zero_playtime_pct = (df_raw['average_forever'] == 0).sum() / len(df_raw) * 100
print(f"Tỷ lệ game có average_playtime = 0: {zero_playtime_pct:.2f}%")
print('Nhận xét: Kể từ sau năm 2023, trung bình playtime đều bằng 0 do API SteamSpy ngừng hỗ trợ tracking cột này. Đây là khuyết thiếu hệ thống MNAR.')

print('\n=== 3. OWNERS COLUMN FORMAT ===')
# Print owners string categories
print(df_raw['owners'].value_counts().head(5))
print('Nhận xét: Owners được lưu dưới dạng chuỗi khoảng phân nhóm phi số (ví dụ: "0 .. 20,000"), cần được parse ra numeric.')

print('\n=== 4. RATING SMALL-SAMPLE BIAS ===')
# Compute raw ratio for demonstration
raw_ratio = df_raw['positive'] / (df_raw['positive'] + df_raw['negative'])
total_ratings = df_raw['positive'] + df_raw['negative']
small_sample = df_raw[total_ratings < 10][['name', 'positive', 'negative']].copy()
small_sample['raw_ratio'] = raw_ratio[total_ratings < 10]
print(small_sample.head(5))
print('Nhận xét: Các game có lượng review cực nhỏ (<10) dễ đạt tỷ lệ đánh giá thô 100% (raw_ratio=1.0), gây nhiễu và thiên lệch khi làm biến target.')

print('\n=== 5. DEVELOPER & PUBLISHER DUPLICATION ===')
# Count missing values in raw developer/publisher from SteamSpy (x) and Store (y)
print(df_raw[['developer_x', 'developer_y', 'publisher_x', 'publisher_y']].isnull().sum())
print('Nhận xét: Dữ liệu nhà phát triển và nhà phát hành bị phân mảnh và có giá trị khuyết ở cả 2 API, cần được hợp nhất để tối ưu.')

=== 1. PRICE UNIT BUG (CENTS vs USD) ===
              price  initialprice
count  1.028400e+04  10284.000000
mean   1.115321e+07    965.546383
std    1.228257e+07   1108.991720
min    0.000000e+00      0.000000
25%    3.000000e+06    299.000000
50%    7.350000e+06    599.000000
75%    1.420000e+07   1299.000000
max    9.990000e+07  19999.000000
Sample raw initialprice values: [2999, 6999, 5999, 6999, 999]
Nhận xét: Giá trị max của initialprice lên đến 19,999.00 USD — lạm phát 100 lần do Store API lưu dạng cents (xu) thay vì USD.

=== 2. PLAYTIME SYSTEMATIC MISSINGNESS (MNAR) ===
           average_forever  median_forever
year_temp                                 
2022                   0.0             0.0
2023                   0.0             0.0
2024                   0.0             0.0
2025                   0.0             0.0
2026                   0.0             0.0
Tỷ lệ game có average_playtime = 0: 100.00%
Nhận xét: Kể từ sau năm 2023, trung bình playtime đều bằng 0 do API S

## Phần 2: Quy trình Tiền xử lý & Làm sạch (Curation Pipeline)

Tiến hành xây dựng và thực thi các bước làm sạch dựa trên chẩn đoán ở Phần 1.

In [3]:
# Step 1: Price Bug Fix & Renaming

print('=== STEP 1: Price Bug Fix & Column Renaming ===')
print(f"Before fix — initialprice range: {df_raw['initialprice'].min():.2f} to {df_raw['initialprice'].max():.2f}")

df_raw['price'] = df_raw['initialprice'] / 100
print(f"After fix — price range: {df_raw['price'].min():.2f} to {df_raw['price'].max():.2f}")

assert (df_raw['price'] >= 0).all(), 'Negative prices detected!'
assert df_raw['price'].max() < 1000, 'Price still too high — bug fix may have failed'
print('Price fix validated: all prices >= 0 and < $1000')

# Rename positive/negative columns
df_raw = df_raw.rename(columns={'positive': 'positive_ratings', 'negative': 'negative_ratings'})
print('Ratings columns renamed to positive_ratings and negative_ratings')

# Merge developer and publisher
df_raw['developer'] = df_raw['developer_y'].fillna(df_raw['developer_x'])
df_raw['publisher'] = df_raw['publisher_y'].fillna(df_raw['publisher_x'])
print('Developer and publisher columns unified')

# Parse owners string
owners_split = df_raw['owners'].str.split(' .. ', expand=True)
df_raw['owners_min'] = pd.to_numeric(owners_split[0].str.replace(',', ''), errors='coerce').astype('Int64')
df_raw['owners_max'] = pd.to_numeric(owners_split[1].str.replace(',', ''), errors='coerce').astype('Int64')
print('Owners range parsed to numeric owners_min and owners_max')

# Parse year early for filtering
df_raw['year'] = pd.to_datetime(df_raw['release_date'], errors='coerce').dt.year.astype('Int64')

# Filter rows strictly to target period (2022-2026) and rating threshold (> 100 total reviews)
total_ratings = df_raw['positive_ratings'] + df_raw['negative_ratings']
before_count = len(df_raw)
df_raw = df_raw[(total_ratings > 100) & (df_raw['year'].between(2022, 2026))].copy()
print(f'Filtered rows: {before_count} -> {len(df_raw)} (retaining only games released 2022-2026 with > 100 ratings)')

=== STEP 1: Price Bug Fix & Column Renaming ===
Before fix — initialprice range: 0.00 to 19999.00
After fix — price range: 0.00 to 199.99
Price fix validated: all prices >= 0 and < $1000
Ratings columns renamed to positive_ratings and negative_ratings
Developer and publisher columns unified
Owners range parsed to numeric owners_min and owners_max
Filtered rows: 10284 -> 3868 (retaining only games released 2022-2026 with > 100 ratings)


In [4]:
# Step 2: Drop playtime (MNAR)
# SteamSpy returns 0 for ALL games after 2023 — technical API limitation
# This is MNAR: missingness depends on the unobserved value itself
# Decision: drop entirely to prevent bias

print('=== STEP 2: MNAR — Drop playtime columns ===')
playtime_cols = [c for c in df_raw.columns if any(x in c.lower() for x in ['playtime', 'median', 'average'])]
print(f'Playtime-related columns to drop: {playtime_cols}')

df_raw = df_raw.drop(columns=playtime_cols, errors='ignore')
print(f'Shape after dropping playtime: {df_raw.shape}')

=== STEP 2: MNAR — Drop playtime columns ===
Playtime-related columns to drop: ['average_forever', 'average_2weeks', 'median_forever', 'median_2weeks']
Shape after dropping playtime: (3868, 26)


In [5]:
# Step 3: Remove redundant/raw intermediate columns

print('=== STEP 3: Remove redundant columns ===')
redundant = [
    'developer_x', 'developer_y', 'publisher_x', 'publisher_y',
    'score_rank', 'userscore',
    'store_name', 'store_price', 'initialprice', 'year_temp'
]
existing = [c for c in redundant if c in df_raw.columns]
df_raw = df_raw.drop(columns=existing)
print(f'Dropped: {existing}')
print(f'Shape after dedup: {df_raw.shape}')

=== STEP 3: Remove redundant columns ===
Dropped: ['developer_x', 'developer_y', 'publisher_x', 'publisher_y', 'score_rank', 'userscore', 'store_name', 'store_price', 'initialprice', 'year_temp']
Shape after dedup: (3868, 16)


In [6]:
# Step 4: Wilson Score Calculation
# Problem: rating_ratio is biased for small samples
# Solution: Wilson Score Interval lower bound (95% confidence, z=1.96)

print('=== STEP 4: Wilson Score Calculation ===')

z = 1.96
p_hat = df_raw['positive_ratings'] / (df_raw['positive_ratings'] + df_raw['negative_ratings'])
n = df_raw['positive_ratings'] + df_raw['negative_ratings']

df_raw['wilson_score'] = (
    (p_hat + z**2 / (2*n) - z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4*n**2)))
    / (1 + z**2 / n)
)

print(f"Wilson score range: [{df_raw['wilson_score'].min():.4f}, {df_raw['wilson_score'].max():.4f}]")

=== STEP 4: Wilson Score Calculation ===
Wilson score range: [0.0529, 0.9946]


In [7]:
# Step 5: Genre encoding
# Parse genres from string representation to list, then create one-hot columns

print('=== STEP 5: Genre Encoding ===')

def parse_genres(val):
    if isinstance(val, str):
        try:
            result = ast.literal_eval(val)
            if isinstance(result, list):
                return result
        except:
            return []
    elif isinstance(val, list):
        return val
    return []

df_raw['genres'] = df_raw['genres'].apply(parse_genres)

all_genres = sorted(set(g for genres in df_raw['genres'] for g in genres))
print(f'Found {len(all_genres)} unique genres')

for genre in all_genres:
    col_name = f'genre_{genre}'
    df_raw[col_name] = df_raw['genres'].apply(lambda x: 1 if genre in x else 0)

genre_cols = [c for c in df_raw.columns if c.startswith('genre_')]
no_genre = (df_raw[genre_cols].sum(axis=1) == 0).sum()
print(f'Games with no genre flags: {no_genre}')
print(f'Shape after genre encoding: {df_raw.shape}')

=== STEP 5: Genre Encoding ===
Found 22 unique genres
Games with no genre flags: 14
Shape after genre encoding: (3868, 39)


In [8]:
# Step 6: Owners MNAR handling
# SteamSpy bucket '0 .. 20,000' has owners_min=0, but this is NOT a real lower bound
# It means 'at least 0' which is meaningless — API doesn't provide lower bound for smallest bucket
# Fix: set owners_min to NaN, add flag column

print('=== STEP 6: Owners MNAR Handling ===')

mask_small = df_raw['owners'] == '0 .. 20,000'
df_raw.loc[mask_small, 'owners_min'] = np.nan
df_raw['owners_min_known'] = ~mask_small

df_raw['owners_midpoint'] = ((df_raw['owners_min'].fillna(0) + df_raw['owners_max']) // 2).astype('Int64')
df_raw.loc[df_raw['owners_min'].isna(), 'owners_midpoint'] = pd.NA

print(f"Games with unknown owners_min: {mask_small.sum()} ({mask_small.sum()/len(df_raw)*100:.1f}%)")

=== STEP 6: Owners MNAR Handling ===
Games with unknown owners_min: 835 (21.6%)


In [9]:
# Step 7: Compute rating_ratio and verify

df_raw['rating_ratio'] = df_raw['positive_ratings'] / (df_raw['positive_ratings'] + df_raw['negative_ratings'])
print(f"rating_ratio range: [{df_raw['rating_ratio'].min():.4f}, {df_raw['rating_ratio'].max():.4f}]")
assert df_raw['rating_ratio'].between(0, 1).all()
print('rating_ratio validated')

# Compute is_free
df_raw['is_free'] = (df_raw['price'] == 0).astype('int64')

# Compute price_group
df_raw['price_group'] = pd.cut(
    df_raw['price'],
    bins=[-1, 0, 10, 30, float('inf')],
    labels=['Free', '<$10', '$10-30', '>$30']
).astype(str)

rating_ratio range: [0.0932, 1.0000]
rating_ratio validated


In [10]:
# Step 8: Column ordering & final cleanup

print('=== STEP 8: Final Column Ordering ===')

core_cols = [
    'appid', 'name', 'release_date', 'year', 'developer', 'publisher',
    'price', 'is_free', 'discount', 'price_group',
    'positive_ratings', 'negative_ratings', 'rating_ratio', 'wilson_score',
    'owners_min', 'owners_max', 'owners_midpoint', 'owners_min_known',
    'ccu', 'genres',
]

genre_cols = sorted([c for c in df_raw.columns if c.startswith('genre_')])
final_cols = core_cols + genre_cols

df_final = df_raw[final_cols].copy()
df_final['release_date'] = pd.to_datetime(df_final['release_date'], errors='coerce')
print(f'Final shape: {df_final.shape}')
print(f'Final columns ({len(df_final.columns)}): {list(df_final.columns)}')
df_final.head(3)

=== STEP 8: Final Column Ordering ===
Final shape: (3868, 42)
Final columns (42): ['appid', 'name', 'release_date', 'year', 'developer', 'publisher', 'price', 'is_free', 'discount', 'price_group', 'positive_ratings', 'negative_ratings', 'rating_ratio', 'wilson_score', 'owners_min', 'owners_max', 'owners_midpoint', 'owners_min_known', 'ccu', 'genres', 'genre_Accounting', 'genre_Action', 'genre_Adventure', 'genre_Animation & Modeling', 'genre_Audio Production', 'genre_Casual', 'genre_Design & Illustration', 'genre_Early Access', 'genre_Education', 'genre_Free To Play', 'genre_Game Development', 'genre_Indie', 'genre_Massively Multiplayer', 'genre_Photo Editing', 'genre_RPG', 'genre_Racing', 'genre_Simulation', 'genre_Sports', 'genre_Strategy', 'genre_Utilities', 'genre_Video Production', 'genre_Web Publishing']


,appid,name,release_date,year,developer,publisher,price,is_free,discount,price_group,...,genre_Massively Multiplayer,genre_Photo Editing,genre_RPG,genre_Racing,genre_Simulation,genre_Sports,genre_Strategy,genre_Utilities,genre_Video Production,genre_Web Publishing
0,1623730,Palworld,2024-01-18,2024,Pocketpair,Pocketpair,29.99,0,0,$10-30,...,0,0,1,0,0,0,0,0,0,0
1,1938090,Call of Duty: Modern Warfare II,2022-10-27,2022,Treyarch,Activision,69.99,0,45,>$30,...,0,0,0,0,0,0,0,0,0,0
2,2358720,Black Myth: Wukong,2024-08-19,2024,Game Science,Game Science,59.99,0,0,>$30,...,0,0,1,0,0,0,0,0,0,0


In [11]:
# Step 9: QA Assertions

print('=== STEP 9: Automated QA Assertions ===')

checks = []

# 1. AppID uniqueness
checks.append({'assertion': 'appid is unique',
    'result': 'PASS' if df_final['appid'].nunique() == len(df_final) else 'FAIL',
    'details': f"{df_final['appid'].nunique()} unique / {len(df_final)} total"})

# 2. Price bounds
checks.append({'assertion': 'price >= 0',
    'result': 'PASS' if (df_final['price'] >= 0).all() else 'FAIL',
    'details': f"Range: [{df_final['price'].min():.2f}, {df_final['price'].max():.2f}]"})

# 3. Rating ratio bounds
checks.append({'assertion': '0 <= rating_ratio <= 1',
    'result': 'PASS' if df_final['rating_ratio'].between(0, 1).all() else 'FAIL',
    'details': f"Range: [{df_final['rating_ratio'].min():.4f}, {df_final['rating_ratio'].max():.4f}]"})

# 4. Wilson score bounds
checks.append({'assertion': '0 <= wilson_score <= 1',
    'result': 'PASS' if df_final['wilson_score'].between(0, 1).all() else 'FAIL',
    'details': f"Range: [{df_final['wilson_score'].min():.4f}, {df_final['wilson_score'].max():.4f}]"})

# 5. Year bounds
checks.append({'assertion': 'year in [2022, 2026]',
    'result': 'PASS' if df_final['year'].between(2022, 2026).all() else 'FAIL',
    'details': f'Years: {sorted(df_final["year"].unique())}'})

# 6. Total ratings threshold
total_r = df_final['positive_ratings'] + df_final['negative_ratings']
checks.append({'assertion': 'total_ratings >= 100',
    'result': 'PASS' if (total_r >= 100).all() else 'FAIL',
    'details': f'Min: {total_r.min()}'})

# 7. Redundant columns removed
redundant_check = not any(c in df_final.columns for c in ['owners', 'total_ratings', 'initialprice', 'store_name'])
checks.append({'assertion': 'redundant columns removed',
    'result': 'PASS' if redundant_check else 'FAIL',
    'details': 'owners, total_ratings, initialprice, store_name all removed'})

# 8. owners_min MNAR flag
mnar_check = df_final[df_final['owners_min_known'] == False]['owners_min'].isna().all()
checks.append({'assertion': 'owners_min NaN for MNAR bucket',
    'result': 'PASS' if mnar_check else 'FAIL',
    'details': "'0..20,000' bucket correctly marked as NaN"})

# 9. is_free/price consistency
free_price_check = ((df_final['is_free'] == 1) & (df_final['price'] > 0)).sum() == 0
checks.append({'assertion': 'is_free=1 implies price=0',
    'result': 'PASS' if free_price_check else 'FAIL',
    'details': 'All free games have price=0'})

checks_df = pd.DataFrame(checks)
print(checks_df.to_string(index=False))
all_pass = (checks_df['result'] == 'PASS').all()
print(f'\n{"ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"}')

=== STEP 9: Automated QA Assertions ===
                     assertion result                                                                                 details
               appid is unique   PASS                                                                3868 unique / 3868 total
                    price >= 0   PASS                                                                    Range: [0.00, 99.99]
        0 <= rating_ratio <= 1   PASS                                                                 Range: [0.0932, 1.0000]
        0 <= wilson_score <= 1   PASS                                                                 Range: [0.0529, 0.9946]
          year in [2022, 2026]   PASS Years: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
          total_ratings >= 100   PASS                                                                                Min: 101
     redundant columns removed   PASS                             owners, tota

In [12]:
# Step 10: Export

print('=== STEP 10: Export Cleaned Dataset ===')

# Save the full 42-column dataset (including derived columns)
df_final.to_csv(PROCESSED_PATH, index=False)
print(f'Saved clean dataset with all columns to: {PROCESSED_PATH}')

print('\nFinal dataset summary:')
print(f'  Records: {len(df_final):,}')
print(f'  Columns: {len(df_final.columns)}')
core_count = len([c for c in df_final.columns if not c.startswith("genre_")])
genre_count = len([c for c in df_final.columns if c.startswith("genre_")])
print(f'  Core columns: {core_count}')
print(f'  Genre columns: {genre_count}')

print('\nColumn list:')
for i, c in enumerate(df_final.columns):
    dtype = df_final[c].dtype
    missing = df_final[c].isnull().sum()
    print(f'  {i:2d}. {c} ({dtype}) — missing: {missing}')

=== STEP 10: Export Cleaned Dataset ===
Saved clean dataset with all columns to: D:\DS108_final\data\processed\steam_games_with_genres.csv

Final dataset summary:
  Records: 3,868
  Columns: 42
  Core columns: 20
  Genre columns: 22

Column list:
   0. appid (int64) — missing: 0
   1. name (object) — missing: 0
   2. release_date (datetime64[ns]) — missing: 0
   3. year (Int64) — missing: 0
   4. developer (object) — missing: 0
   5. publisher (object) — missing: 5
   6. price (float64) — missing: 0
   7. is_free (int64) — missing: 0
   8. discount (int64) — missing: 0
   9. price_group (object) — missing: 0
  10. positive_ratings (int64) — missing: 0
  11. negative_ratings (int64) — missing: 0
  12. rating_ratio (float64) — missing: 0
  13. wilson_score (float64) — missing: 0
  14. owners_min (Int64) — missing: 835
  15. owners_max (Int64) — missing: 0
  16. owners_midpoint (Int64) — missing: 835
  17. owners_min_known (bool) — missing: 0
  18. ccu (int64) — missing: 0
  19. genres (o